# Question 4 — Linear Regression (MrBeast YouTube Data)
**请将 `mr_beast.csv` 放在同一目录后运行所有单元格**

In [ ]:
# ============================================================
# 导入库
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# Q1: 读入数据，将 publish_day 转为有序 Categorical，
#     然后用 data.sample() 90/10 分割（random_state=617）。
#     问：train 样本中 likes 的中位数是多少？
# ============================================================

data = pd.read_csv('mr_beast.csv')

data['publish_day'] = pd.Categorical(
    data['publish_day'],
    categories=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'],
    ordered=True
)

# 90% train
train = data.sample(frac=0.9, random_state=617)
test  = data.drop(labels=train.index)

print('Train 样本量:', len(train))
print('Test  样本量:', len(test))
print()
print('Q1 答案 — train 中 likes 的中位数:', train['likes'].median())

In [ ]:
# ============================================================
# Q2: 绘制 views (x轴) vs likes (y轴) 的散点图。
#     问：点的方向是什么？
# ============================================================

plt.figure(figsize=(8, 5))
plt.scatter(train['views'], train['likes'], alpha=0.6, edgecolors='k', linewidths=0.3)
plt.xlabel('Views')
plt.ylabel('Likes')
plt.title('Views vs Likes (MrBeast)')
plt.tight_layout()
plt.show()

print('Q2 答案 — 点的方向: POSITIVE（正向）')
print('  → views 越多，likes 越多，呈正相关。')

In [ ]:
# ============================================================
# Q3: 哪些变量与 likes 有统计显著的 Pearson 相关？
#     （显著 = p < 0.05）
# ============================================================

numeric_cols = ['views', 'comments', 'duration_seconds',
                'number_of_tags', 'length_description',
                'length_title', 'time_since']

print('Q3 — 各变量与 likes 的 Pearson 相关及 p 值:')
print(f"{'变量':<22} {'r':>8} {'p-value':>12} {'显著?':>8}")
print('-' * 55)
for col in numeric_cols:
    r, p = stats.pearsonr(train[col].dropna(), train['likes'][train[col].notna()])
    sig = '✅ 是' if p < 0.05 else '❌ 否'
    print(f'{col:<22} {r:>8.4f} {p:>12.4f} {sig:>8}')

In [ ]:
# ============================================================
# Q4: 与 likes 相关系数（绝对值）最强的变量是哪个？
# ============================================================

print('Q4 — 相关系数绝对值排名:')
corrs = {}
for col in numeric_cols:
    r, _ = stats.pearsonr(train[col].dropna(), train['likes'][train[col].notna()])
    corrs[col] = abs(r)

corr_series = pd.Series(corrs).sort_values(ascending=False)
print(corr_series.round(4))
print()
print('Q4 答案 — 最强相关:', corr_series.index[0], f'(|r| = {corr_series.iloc[0]:.4f})')

In [ ]:
# ============================================================
# Q5: 用 time_since 预测 likes，构建 model1（statsmodels）。
#     time_since 的 p 值是多少？系数是否显著？
# Q6: time_since 系数的含义是什么？
# ============================================================

model1 = smf.ols('likes ~ time_since', data=train).fit()
print(model1.summary())
print()

b_time = model1.params['time_since']
p_time = model1.pvalues['time_since']

print('=' * 50)
print(f'Q5 答案 — time_since 系数: {b_time:.4f}')
print(f'        p-value         : {p_time:.6f}')
if p_time < 0.05:
    print('        → p < 0.05，系数显著！time_since 对 likes 有显著影响。')
else:
    print('        → p >= 0.05，系数不显著。')

print()
if b_time > 0:
    print(f'Q6 答案 — 每增加 1 小时，likes 平均增加 {b_time:.2f} 个。')
else:
    print(f'Q6 答案 — 每增加 1 小时，likes 平均减少 {abs(b_time):.2f} 个。')

In [ ]:
# ============================================================
# Q7: 第一个观测（video_id='gL6iSCSHjco'）实际有 2,125,003 likes。
#     用 model1 预测其 likes 是多少？
# ============================================================

first_video = train[train['video_id'] == 'gL6iSCSHjco']

if len(first_video) == 0:
    print('⚠️  未在 train 中找到 video_id=gL6iSCSHjco，请检查数据。')
    print('   尝试用 train.iloc[[0]] 代替:')
    first_video = train.iloc[[0]]
    print('   使用第一行，video_id =', train.iloc[0]['video_id'])

pred_first = model1.predict(first_video)
print(f'Q7 答案 — 第一个视频的预测 likes: {pred_first.values[0]:,.0f}')
print(f'         实际 likes: 2,125,003')
print(f'         残差: {2125003 - pred_first.values[0]:,.0f}')

In [ ]:
# ============================================================
# Q8: 根据 model1，一个已发布 25,000 小时的视频预测 likes 是多少？
# ============================================================

new_obs = pd.DataFrame({'time_since': [25000]})
pred_25k = model1.predict(new_obs)
print(f'Q8 答案 — time_since=25000 小时的预测 likes: {pred_25k.values[0]:,.0f}')

In [ ]:
# ============================================================
# Q9: 构建 model2（number_of_tags → likes）。
#     每多加一个 tag，likes 平均变化多少？
# ============================================================

model2 = smf.ols('likes ~ number_of_tags', data=train).fit()
print(model2.summary())
print()

b_tags = model2.params['number_of_tags']
p_tags = model2.pvalues['number_of_tags']

print('=' * 50)
print(f'Q9 答案 — number_of_tags 系数: {b_tags:.4f}')
print(f'         p-value: {p_tags:.4f}')
if b_tags > 0:
    print(f'         → 每多一个 tag，likes 平均增加 {b_tags:,.2f} 个。')
else:
    print(f'         → 每多一个 tag，likes 平均减少 {abs(b_tags):,.2f} 个。')

In [ ]:
# ============================================================
# Q10: 构建 model3（publish_day → likes）。R² 是多少？
# Q11: 根据 model3，哪些陈述是正确的？
# Q12: 周四发布的视频比周一平均多多少 likes？
# ============================================================

# 注意：publish_day 是 Categorical，statsmodels 会自动生成虚拟变量
# Baseline = Mon（第一个 category）
model3 = smf.ols('likes ~ C(publish_day)', data=train).fit()
print(model3.summary())
print()

r2_m3 = model3.rsquared
print(f'Q10 答案 — model3 R²: {r2_m3:.4f}')
print()

# 各天系数
params = model3.params
pvals  = model3.pvalues
print('各天系数（相对于 Monday）及 p 值:')
for k in params.index:
    print(f'  {k:<35} coef={params[k]:>12.2f}  p={pvals[k]:.4f}')

print()
# Q12: Thursday coefficient
thu_key = [k for k in params.index if 'Thu' in k]
if thu_key:
    thu_coef = params[thu_key[0]]
    print(f'Q12 答案 — 周四比周一平均多 likes: {thu_coef:,.2f}')
    if thu_coef > 0:
        print(f'          即周四视频平均比周一多 {thu_coef:,.0f} 个 likes。')
    else:
        print(f'          即周四视频平均比周一少 {abs(thu_coef):,.0f} 个 likes。')
else:
    print('⚠️  未找到 Thursday 系数，请检查变量名称。')

In [ ]:
# ============================================================
# Q13: sklearn 框架
#      y = likes, X = [duration_seconds, number_of_tags,
#                       length_description, length_title, time_since, publish_day]
#      y_binned = pd.cut(data.likes, bins=[0,1000,10000,100000,1000000,25000000])
#      stratified 90/10 split, random_state=617
#      问：train 样本中 duration_seconds 的中位数是多少？
# ============================================================

features = ['duration_seconds', 'number_of_tags', 'length_description',
            'length_title', 'time_since', 'publish_day']

y = data['likes']
X = data[features]

y_binned = pd.cut(data.likes, bins=[0, 1000, 10000, 100000, 1000000, 25000000])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.9, random_state=617, stratify=y_binned)

print('Stratified train 样本量:', len(X_train))
print('Stratified test  样本量:', len(X_test))
print()
print('Q13 答案 — train 中 duration_seconds 的中位数:', X_train['duration_seconds'].median())

In [ ]:
# ============================================================
# Q14: 对 publish_day 进行 dummy coding（drop_first=True）。
#      Tuesday 的虚拟变量均值是多少？
# ============================================================

# train
X_train_d = pd.get_dummies(X_train, columns=['publish_day'], drop_first=True)
X_test_d  = pd.get_dummies(X_test,  columns=['publish_day'], drop_first=True)

# 对齐 test 到 train 的列
X_test_d = X_test_d.reindex(columns=X_train_d.columns, fill_value=0)

print('Dummy coding 后的列名:')
print(X_train_d.columns.tolist())
print()

# 找到 Tuesday 的列名
tue_col = [c for c in X_train_d.columns if 'Tue' in c]
if tue_col:
    tue_mean = X_train_d[tue_col[0]].mean()
    print(f'Q14 答案 — Tuesday dummy 变量的均值: {round(tue_mean, 4)}')
    print(f'          ({tue_col[0]})')
else:
    print('⚠️  未找到 Tuesday dummy 变量，请检查列名。')

In [ ]:
# ============================================================
# Q15: 拟合 model4（sklearn LinearRegression），预测所有特征 → likes。
#      train 最后一个观测的预测 likes 是多少？
# ============================================================

model4 = LinearRegression()
model4.fit(X_train_d, y_train)

y_pred_train = model4.predict(X_train_d)
y_pred_test  = model4.predict(X_test_d)

pred_last = y_pred_train[-1]
actual_last = y_train.iloc[-1]

print(f'Q15 答案 — train 最后一个观测的预测 likes: {pred_last:,.2f}')
print(f'          实际 likes: {actual_last:,}')

In [ ]:
# ============================================================
# Q16: model4 在 TRAIN 样本的 RMSE 是多少？
# ============================================================

rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
print(f'Q16 答案 — model4 Train RMSE: {rmse_train:,.2f}')

In [ ]:
# ============================================================
# Q17: model4 在 TEST 样本的 RMSE 是多少？
# ============================================================

rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
print(f'Q17 答案 — model4 Test RMSE: {rmse_test:,.2f}')
print()

# 综合对比
print('=' * 45)
print('汇总:')
print(f'  Train RMSE: {rmse_train:>15,.2f}')
print(f'  Test  RMSE: {rmse_test:>15,.2f}')
if rmse_test > rmse_train * 1.2:
    print('  → Test RMSE 明显高于 Train RMSE，可能存在过拟合。')
else:
    print('  → Train ≈ Test RMSE，模型泛化良好。')

---
## 📋 答案速查（运行代码后核对）

| 题目 | 内容 | 答案 |
|---|---|---|
| Q1 | train 中 likes 的中位数 | **运行 Q1 单元格** |
| Q2 | scatter plot 方向 | **Positive（正向）** |
| Q3 | 与 likes 显著相关的变量 | **运行 Q3（p<0.05 的变量）** |
| Q4 | 与 likes 相关最强的变量 | **运行 Q4** |
| Q5 | time_since 的 p 值 | **运行 Q5（通常 p<0.05）** |
| Q6 | time_since 系数含义 | **每+1小时，likes 增加 coef 个** |
| Q7 | 第一个视频的预测 likes | **运行 Q7** |
| Q8 | 25000 小时视频的预测 likes | **运行 Q8** |
| Q9 | number_of_tags 系数 | **运行 Q9** |
| Q10 | model3 R² | **运行 Q10** |
| Q11 | model3 哪些陈述正确 | **看各天系数和 p 值** |
| Q12 | Thursday vs Monday 差异 | **运行 Q12（Thu 的系数）** |
| Q13 | duration_seconds 中位数 | **运行 Q13** |
| Q14 | Tuesday dummy 均值 | **运行 Q14** |
| Q15 | 最后一个观测的预测值 | **运行 Q15** |
| Q16 | Train RMSE | **运行 Q16** |
| Q17 | Test RMSE | **运行 Q17** |

---
## 🧠 关键知识点

| 概念 | 说明 |
|---|---|
| **p-value < 0.05** | 系数在统计上显著，即变量对因变量有影响 |
| **正相关系数** | X 增加 → Y 增加 |
| **R²** | 模型解释的方差比例，0–1，越高越好 |
| **Dummy coding** | 分类变量转为 0/1，drop_first=True 避免多重共线性 |
| **Reference level** | drop_first 后被删除的类别（本题为 Mon） |
| **RMSE** | √(MSE)，单位与 Y 相同，越低越好 |
| **Stratified split** | 按 y_binned 分层，保持各类别比例 |